# MonsoonCast: Notebook 02 — Model Verification & Error Analysis

**Author:** Khushaal Srivastava  
**Project:** MonsoonCast  
**Date:** February 23, 2026  

This notebook analyzes model predictions on the held-out test partition (2021–2023), evaluates probability calibration curves, inspects false alarm vs miss trade-offs, and diagnoses spatial error distributions.

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to Python path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from monsooncast.evaluation.metrics import compute_classification_metrics, compute_probabilistic_metrics
from monsooncast.evaluation.reliability import compute_reliability_curve

## 1. Load Evaluation Results & Metrics

In [ ]:
metrics_file = Path("../artifacts/metrics/evaluation_results.json")
if metrics_file.exists():
    with open(metrics_file, "r") as f:
        results = json.load(f)
    df_res = pd.DataFrame(results).T
    print(df_res[["pr_auc", "roc_auc", "pod", "far", "csi", "brier_skill_score", "ece"]])
else:
    print("Metrics file not found. Run 'python scripts/evaluate.py' first.")

## 2. Compare Reliability Diagrams Across Models

In [ ]:
rel_file = Path("../artifacts/metrics/reliability_curves.json")
if rel_file.exists():
    with open(rel_file, "r") as f:
        rel_data = json.load(f)
    
    plt.figure(figsize=(7, 6))
    plt.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
    
    for m_name, d in rel_data.items():
        plt.plot(d["bin_means"], d["bin_fracs"], marker="o", label=f"{m_name} (ECE={d.get('ece', 0):.3f})")
        
    plt.title("Reliability Diagrams on Held-Out Test Set (2021-2023)")
    plt.xlabel("Forecast Probability")
    plt.ylabel("Observed Event Frequency")
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.show()